In [1]:
import sys
from pathlib import Path

# Add the project root to the path
sys.path.insert(0, str(Path.cwd().parent.parent.parent))

from src.constants import EYE_BY_WORD_ALIGNED_PATHS
from src.Eye_metrics.eye_df_utils import get_eye_df
from loguru import logger
import pandas as pd

aligned_path = EYE_BY_WORD_ALIGNED_PATHS['L1_and_L2']

In [2]:
# load eye_df with aligned_idx
logger.info("Loading eye_df...")
eye_df = get_eye_df(aligned_path, L1_or_L2="L1_and_L2")
eye_df = eye_df[eye_df['reread']==0]
print(f"eye_df n rows without reread: {len(eye_df)}")
eye_df.head()

2026-06-29 12:06:37.146 | INFO     | __main__:<module>:2 - Loading eye_df...
2026-06-29 12:06:37.146 | INFO     | src.Eye_metrics.eye_df_utils:get_eye_df:56 - Preprocess| Loading L1_and_L2 word df...
2026-06-29 12:06:57.009 | INFO     | src.Eye_metrics.eye_df_utils:get_eye_df:58 - Preprocess| n rows L1_and_L2 word df: 4479260


eye_df n rows without reread: 3733817


,,subject_id,TRIAL_INDEX,EYE_REPORTED,EYE_TRACKED,GROUPING_VARIABLES,IA_AREA,IA_AVERAGE_FIX_PUPIL_SIZE,IA_BOTTOM,TF,...,is_correct,text_id,fill_notFirstFixProg,FirstPassGD,FirstPassFF,GoPast,align_idx,sentence_idx,L1_or_L2,Reduced_POS
0,0,l42_2070,4,RIGHT,Right,RECORDING_SESSION,10545.0,1287.00,264,216,...,True,3_6_1,NaN,216.0,216.0,216.0,744,1.0,L1,None
1,1,l42_2070,4,RIGHT,Right,RECORDING_SESSION,12654.0,1267.75,264,1217,...,True,3_6_1,NaN,NaN,NaN,NaN,744,1.0,L1,None
2,2,l42_2070,4,RIGHT,Right,RECORDING_SESSION,12654.0,1273.00,264,173,...,True,3_6_1,NaN,173.0,173.0,448.0,744,1.0,L1,None
3,3,l42_2070,4,RIGHT,Right,RECORDING_SESSION,12654.0,NaN,264,0,...,True,3_6_1,NaN,NaN,NaN,NaN,744,1.0,L1,None
4,4,l42_2070,4,RIGHT,Right,RECORDING_SESSION,12654.0,NaN,264,0,...,True,3_6_1,NaN,NaN,NaN,NaN,744,1.0,L1,None


## N subjects L1 and L2

In [3]:
eye_df[['subject_id','L1_or_L2']].drop_duplicates().value_counts(subset='L1_or_L2')

L1_or_L2
L1    360
L2    278
Name: count, dtype: int64

## N Subjects Hunting and gathering

In [4]:
eye_df[['subject_id','has_preview']].drop_duplicates().value_counts(subset='has_preview')

has_preview
Gathering    334
Hunting      304
Name: count, dtype: int64

# N word tokens overall

In [5]:
print(f"{eye_df.shape[0]:,}")

3,733,817


## N word tokens per text level

In [6]:
print(f"{eye_df.groupby('level').size()}")

level
Adv    2063741
Ele    1670076
dtype: int64


# N Subjects per text

In [27]:
n_subjects_per_text = eye_df.groupby(['text_id','has_preview','level'])['subject_id'].nunique().reset_index().rename(columns={'subject_id':'n_subjects'})
n_subjects_per_text

,text_id,has_preview,level,n_subjects
0,1_10_1,Gathering,Adv,59
1,1_10_1,Gathering,Ele,58
2,1_10_1,Hunting,Adv,55
3,1_10_1,Hunting,Ele,56
4,1_10_2,Gathering,Adv,58
...,...,...,...,...
643,3_9_3,Hunting,Ele,47
644,3_9_4,Gathering,Adv,51
645,3_9_4,Gathering,Ele,55
646,3_9_4,Hunting,Adv,47


In [33]:
# 1) Sum n_subjects across levels for each (text_id)
totals_per_text_preview = (
    n_subjects_per_text
    .groupby(['text_id'], as_index=False)['n_subjects']
    .sum()
    .rename(columns={'n_subjects': 'total_n_subjects'})
)

# 2) Mean of those totals within each text_id
mean_per_text = (
    totals_per_text_preview
    .groupby('text_id', as_index=False)['total_n_subjects']
    .mean()
    .rename(columns={'total_n_subjects': 'mean_n_subjects_per_text'})
)

# 3) Overall mean across text_id
overall_mean = mean_per_text['mean_n_subjects_per_text'].mean()

print("n partciapants per text, not splitted by has_preview")
overall_mean

n partciapants per text, not splitted by has_preview


212.5493827160494

In [32]:
# 1) Sum n_subjects across levels for each (text_id, level) — keep level
totals_per_text_preview = (
    n_subjects_per_text
    .groupby(['text_id', 'level'], as_index=False)['n_subjects']
    .sum()
    .rename(columns={'n_subjects': 'total_n_subjects'})
)

# 2) Mean of those totals within each (text_id, level)
mean_per_text = (
    totals_per_text_preview
    .groupby(['text_id', 'level'], as_index=False)['total_n_subjects']
    .mean()
    .rename(columns={'total_n_subjects': 'mean_n_subjects_per_text'})
)

# 3) Overall mean across text_id and level
overall_mean = mean_per_text.groupby('level')['mean_n_subjects_per_text'].mean()
overall_mean

level
Adv    106.296296
Ele    106.253086
Name: mean_n_subjects_per_text, dtype: float64

## Old

In [10]:
eye_df.groupby('L1_or_L2').size()

L1_or_L2
L1    2110632
L2    1623185
dtype: int64

In [11]:
eye_df.groupby('has_preview').size()

has_preview
Gathering    1954605
Hunting      1779212
dtype: int64

In [12]:
eye_df.groupby('level').size()

level
Adv    2063741
Ele    1670076
dtype: int64

In [13]:
n_subjects_per_item = eye_df[eye_df['reread']==0].groupby(['has_preview', 'unique_paragraph_id', 'level'])['subject_id'].nunique().reset_index().rename(columns={'subject_id':'n_subjects'})
n_subjects_per_item

,has_preview,unique_paragraph_id,level,n_subjects
0,Gathering,1_10_Adv_1,Adv,59
1,Gathering,1_10_Adv_2,Adv,58
2,Gathering,1_10_Adv_3,Adv,59
3,Gathering,1_10_Adv_4,Adv,59
4,Gathering,1_10_Adv_5,Adv,59
...,...,...,...,...
643,Hunting,3_9_Adv_4,Adv,47
644,Hunting,3_9_Ele_1,Ele,47
645,Hunting,3_9_Ele_2,Ele,47
646,Hunting,3_9_Ele_3,Ele,47


In [14]:
n_subjects_per_item.groupby(['has_preview', 'level'])['n_subjects'].mean()

has_preview  level
Gathering    Adv      55.635802
             Ele      55.586420
Hunting      Adv      50.660494
             Ele      50.666667
Name: n_subjects, dtype: float64

In [15]:
n_subjects_per_item.groupby(['level'])['n_subjects'].mean()

level
Adv    53.148148
Ele    53.126543
Name: n_subjects, dtype: float64

In [16]:
n_subjects_per_text = eye_df[eye_df['reread']==0].groupby(['text_id'])['subject_id'].nunique().reset_index().rename(columns={'subject_id':'n_subjects'})
n_subjects_per_text

,text_id,n_subjects
0,1_10_1,228
1,1_10_2,228
2,1_10_3,228
3,1_10_4,228
4,1_10_5,228
...,...,...
157,3_8_6,201
158,3_9_1,200
159,3_9_2,200
160,3_9_3,200


In [17]:
n_subjects_per_text['n_subjects'].describe()

count    162.000000
mean     212.549383
std       11.434421
min      200.000000
25%      201.000000
50%      209.000000
75%      228.000000
max      228.000000
Name: n_subjects, dtype: float64

In [18]:
n_subjects_per_text_level = eye_df[eye_df['reread']==0].groupby(['text_id', 'level'])['subject_id'].nunique().reset_index().rename(columns={'subject_id':'n_subjects'})
n_subjects_per_text_level

,text_id,level,n_subjects
0,1_10_1,Adv,114
1,1_10_1,Ele,114
2,1_10_2,Adv,114
3,1_10_2,Ele,114
4,1_10_3,Adv,114
...,...,...,...
319,3_9_2,Ele,98
320,3_9_3,Adv,102
321,3_9_3,Ele,98
322,3_9_4,Adv,98


In [19]:
n_subjects_per_text_level.groupby('level')['n_subjects'].describe()

,count,mean,std,min,25%,50%,75%,max
level,,,,,,,,
Adv,162.0,106.296296,5.785921,98.0,102.0,104.0,114.0,114.0
Ele,162.0,106.253086,5.818488,98.0,102.0,104.5,114.0,114.0


In [20]:
words = eye_df[['unique_paragraph_id', 'IA_ID', 'IA_LABEL']].drop_duplicates().sort_values(['unique_paragraph_id', 'IA_ID'])
words

,unique_paragraph_id,IA_ID,IA_LABEL
230214,1_10_Adv_1,0,Agios
230215,1_10_Adv_1,1,Efstratios
230216,1_10_Adv_1,2,is
230217,1_10_Adv_1,3,so
230218,1_10_Adv_1,4,"remote,"
...,...,...,...
5407,3_9_Ele_4,73,to
5408,3_9_Ele_4,74,17
5409,3_9_Ele_4,75,do
5410,3_9_Ele_4,76,dangerous


In [21]:
words = eye_df[['unique_paragraph_id', 'IA_ID', 'level']].drop_duplicates().sort_values(['unique_paragraph_id', 'IA_ID'])
words

,unique_paragraph_id,IA_ID,level
230214,1_10_Adv_1,0,Adv
230215,1_10_Adv_1,1,Adv
230216,1_10_Adv_1,2,Adv
230217,1_10_Adv_1,3,Adv
230218,1_10_Adv_1,4,Adv
...,...,...,...
5407,3_9_Ele_4,73,Ele
5408,3_9_Ele_4,74,Ele
5409,3_9_Ele_4,75,Ele
5410,3_9_Ele_4,76,Ele
